In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [2]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


# Active Users Month Over Month

**Difficulty:** Medium
**Category:** Data Analysis / Window Functions
**Technologies:** PySpark, SQL, Pandas

---

## Problem Statement

You are given a DataFrame named `logins` containing user login activity.

The DataFrame contains the following columns:

| Column       | Type     | Description                                              |
| ------------ | -------- | -------------------------------------------------------- |
| `user_id`    | `int`    | Unique identifier of the user                            |
| `login_date` | `string` | Date on which the user logged in, in `YYYY-MM-DD` format |

A user can log in **multiple times on the same day** or on **multiple days within the same month**.

Your task is to calculate the number of **distinct active users for each month** and then calculate the **month-over-month (MoM) percentage change** in active users.

---

## Requirements

### 1. Calculate Monthly Active Users

For each month, count the number of **distinct users** who logged in during that month.

For example:

```text
January:
user 1
user 2
user 3
user 1
user 4
```

The number of active users is:

```text
4
```

because `user_id = 1` logged in multiple times but should only be counted once.

---

### 2. Calculate Month-over-Month Change

For every month after the first month, calculate:

```text
mom_change_pct =
((current_month_users - previous_month_users)
 / previous_month_users) * 100
```

Round the result to **1 decimal place**.

### Example

If January has 100 active users and February has 120:

```text
((120 - 100) / 100) * 100
= 20.0%
```

Therefore:

```text
February → 20.0
```

---

## First Month

The first month does not have a previous month for comparison.

Therefore:

```text
mom_change_pct = NULL
```

---

## Expected Output

Return a DataFrame containing the following columns:

| Column           | Description                                      |
| ---------------- | ------------------------------------------------ |
| `month`          | Month in `YYYY-MM` format                        |
| `active_users`   | Number of distinct active users during the month |
| `mom_change_pct` | Percentage change from the previous month        |

The result must be sorted by:

```text
month ASC
```

---

## Example

### Input

```text
+---------+------------+
| user_id | login_date |
+---------+------------+
|    1    | 2023-01-05 |
|    2    | 2023-01-10 |
|    3    | 2023-01-15 |
|    4    | 2023-01-20 |
|    5    | 2023-01-25 |
|    1    | 2023-02-01 |
|    2    | 2023-02-05 |
|    3    | 2023-02-10 |
|    6    | 2023-02-15 |
|    7    | 2023-02-20 |
|    8    | 2023-02-25 |
|    1    | 2023-03-01 |
|    2    | 2023-03-05 |
|    9    | 2023-03-10 |
|   10    | 2023-03-15 |
+---------+------------+
```

### Step 1 — Monthly Active Users

January:

```text
Users = {1, 2, 3, 4, 5}
Active users = 5
```

February:

```text
Users = {1, 2, 3, 6, 7, 8}
Active users = 6
```

March:

```text
Users = {1, 2, 9, 10}
Active users = 4
```

Therefore:

```text
January → 5
February → 6
March → 4
```

---

### Step 2 — Calculate MoM Change

For February:

```text
((6 - 5) / 5) * 100
= 20.0%
```

For March:

```text
((4 - 6) / 6) * 100
= -33.3%
```

---

## Expected Output

```text
+---------+--------------+----------------+
| month   | active_users | mom_change_pct |
+---------+--------------+----------------+
| 2023-01 |      5       |      NULL      |
| 2023-02 |      6       |      20.0      |
| 2023-03 |      4       |     -33.3      |
+---------+--------------+----------------+
```

---

## Key Concepts Tested

This problem tests the following data-engineering concepts:

* Date extraction and formatting
* `COUNT(DISTINCT user_id)`
* Grouping and aggregation
* Window functions
* `LAG`
* Month-over-month calculations
* Handling `NULL` values
* Percentage calculations
* Rounding
* Sorting results chronologically

---

## Recommended Approach

A typical solution can be broken into the following steps:

1. Convert `login_date` from string to a date.
2. Extract the month in `YYYY-MM` format.
3. Count distinct `user_id` for each month.
4. Use the `LAG` window function to retrieve the previous month's active-user count.
5. Calculate the MoM percentage change.
6. Set the first month's percentage change to `NULL`.
7. Round the percentage to 1 decimal place.
8. Sort the final result by month ascending.

### Important

The calculation must use **distinct users per month**, not the total number of login records.

For example, if a user logs in 10 times during January, that user contributes:

```text
1 active user
```

not:

```text
10 active users
```


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F



# Sample data
data = [
    (1, "2023-01-05"),
    (2, "2023-01-10"),
    (3, "2023-01-15"),
    (4, "2023-01-20"),
    (5, "2023-01-25"),

    (1, "2023-02-01"),
    (2, "2023-02-05"),
    (3, "2023-02-10"),
    (6, "2023-02-15"),
    (7, "2023-02-20"),
    (8, "2023-02-25"),

    (1, "2023-03-01"),
    (2, "2023-03-05"),
    (9, "2023-03-10"),
    (10, "2023-03-15")
]

# Define schema
columns = [
    "user_id",
    "login_date"
]

# Create DataFrame
logins = spark.createDataFrame(data, columns)

# Convert login_date from string to date
logins = logins.withColumn(
    "login_date",
    F.to_date("login_date", "yyyy-MM-dd")
)

# Display
logins.show()

+-------+----------+
|user_id|login_date|
+-------+----------+
|      1|2023-01-05|
|      2|2023-01-10|
|      3|2023-01-15|
|      4|2023-01-20|
|      5|2023-01-25|
|      1|2023-02-01|
|      2|2023-02-05|
|      3|2023-02-10|
|      6|2023-02-15|
|      7|2023-02-20|
|      8|2023-02-25|
|      1|2023-03-01|
|      2|2023-03-05|
|      9|2023-03-10|
|     10|2023-03-15|
+-------+----------+



In [30]:
from pyspark.sql.functions import *

active_df = logins\
    .groupBy(
        date_format(col("login_date"),"yyyy-MM").alias("month")
            )\
    .agg(
        countDistinct(col("user_id")).alias("active_users")
        )
  

In [31]:
active_df.show()

+-------+------------+
|  month|active_users|
+-------+------------+
|2023-01|           5|
|2023-02|           6|
|2023-03|           4|
+-------+------------+



In [34]:
from pyspark.sql.window import Window

window_specs = Window.orderBy(col("month"))

active_df\
    .withColumn(
        "prev_users",lag("active_users")\
                        .over(window_specs)
                )\
    .withColumn(
        "mom_change_pct",
        round(
            (
                (col("active_users") - col("prev_users"))
                / col("prev_users")
            ) * 100,
            1
        )
    ).show()


+-------+------------+----------+--------------+
|  month|active_users|prev_users|mom_change_pct|
+-------+------------+----------+--------------+
|2023-01|           5|      NULL|          NULL|
|2023-02|           6|         5|          20.0|
|2023-03|           4|         6|         -33.3|
+-------+------------+----------+--------------+

